In [ ]:
%pip install timm==0.9.12 torch torchmetrics torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade tqdm opencv-python pillow --upgrade

In [1]:
import torch
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0)) 

GPU name: NVIDIA GeForce RTX 4060 Ti


In [2]:
# Cell 1
from pathlib import Path
import hashlib, cv2, random, time
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import timm
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast
import json
import optuna
from optuna.pruners import MedianPruner
from torch.utils.data import Subset
import gc
from sklearn.metrics import fbeta_score

# ⚙️ SET YOUR DATA ROOT
DATA_ROOT = Path(r"G:/My Drive/CLPD-MF-Dataset")  
assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
print("Found dataset root:", DATA_ROOT)

c:\Users\Mohamed Hazem\anaconda3\envs\dlclass\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found dataset root: G:\My Drive\CLPD-MF-Dataset


In [3]:
# Cell 2
def list_images_and_labels(root):
    """
    List all images with labels and magnifications from the new folder structure.
    
    Structure:
    G:/My Drive/CLPD-MF-Dataset/
    ├── MF/
    │   └── Patient Name/
    │       ├── x5/
    │       ├── x10/
    │       └── x20/
    └── Non-MF/
        ├── B cell Lymphoma/
        │   └── Patient Name/
        │       ├── x5/
        │       ├── x10/
        │       └── x20/
        ├── PLEVA-PLC/
        └── pseudolymphoma/
    """
    rows = []
    root = Path(root)
    
    # Process MF folder
    mf_dir = root / "MF"
    if mf_dir.exists():
        for patient_dir in mf_dir.iterdir():
            if not patient_dir.is_dir(): continue
            
            patient_name = patient_dir.name
            # if 10x does not exist, print patient name
            if not (patient_dir / 'x10').exists():
                print(f"Warning: No x10 folder for patient {patient_name} in MF")

            # Look for x10 and x20 subfolders
            for mag in ['x10', 'x20']:
                mag_dir = patient_dir / mag
                if mag_dir.exists() and mag_dir.is_dir():
                    # Find all .tif images in this magnification folder
                    for img_path in mag_dir.glob('*.tif'):
                        rows.append({
                            'path': img_path,
                            'label': 'MF',
                            'patient': patient_name,
                            'mag': mag,
                            'subtype': None  # MF has no subtype
                        })
    
    # Process Non-MF folder with subtypes
    nonmf_dir = root / "Non-MF"
    if nonmf_dir.exists():
        # Each subfolder is a disease subtype
        for subtype_dir in nonmf_dir.iterdir():
            if not subtype_dir.is_dir(): continue
            
            subtype = subtype_dir.name  # B cell Lymphoma, PLEVA-PLC, or pseudolymphoma
            
            # Each patient within the subtype
            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir(): continue
                
                patient_name = patient_dir.name
                # Look for x10 and x20 subfolders
                for mag in ['x10', 'x20']:
                    mag_dir = patient_dir / mag
                    if mag_dir.exists() and mag_dir.is_dir():
                        # if dir is empty, print patient name
                        if not any(mag_dir.iterdir()):
                            print(f"Warning: No images found for patient {patient_name} in subtype {subtype} at magnification {mag}")
                        # Find all .tif images in this magnification folder
                        for img_path in mag_dir.glob('*.tif'):
                            rows.append({
                                'path': img_path,
                                'label': 'Non-MF',
                                'patient': patient_name,
                                'mag': mag,
                                'subtype': subtype
                            })
            
    
    return rows

# Load all images
print("\n" + "="*80)
print("LOADING DATASET")
print("="*80 + "\n")

all_images = list_images_and_labels(DATA_ROOT)

print(f"Total images found: {len(all_images)}")
print(f"\nMagnification distribution:")
mag_counts = Counter([r['mag'] for r in all_images])
for mag, count in sorted(mag_counts.items()):
    print(f"  {mag}: {count} images")

print(f"\nLabel distribution:")
label_counts = Counter([r['label'] for r in all_images])
for label, count in sorted(label_counts.items()):
    print(f"  {label}: {count} images")

# Count unique patients
unique_patients = len(set(r['patient'] for r in all_images))
mf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'MF'))
nonmf_patients = len(set(r['patient'] for r in all_images if r['label'] == 'Non-MF'))
print(f"\nUnique patients:")
print(f"  Total: {unique_patients}")
print(f"  MF: {mf_patients}")
print(f"  Non-MF: {nonmf_patients}")

# Show Non-MF subtypes distribution
print(f"\nNon-MF subtypes:")
nonmf_images = [r for r in all_images if r['label'] == 'Non-MF']
subtype_counts = Counter([r['subtype'] for r in nonmf_images])
for subtype, count in sorted(subtype_counts.items()):
    subtype_patients = len(set(r['patient'] for r in nonmf_images if r['subtype'] == subtype))
    print(f"  {subtype}: {count} images ({subtype_patients} patients)")




LOADING DATASET

Total images found: 3755

Magnification distribution:
  x10: 1287 images
  x20: 2468 images

Label distribution:
  MF: 2433 images
  Non-MF: 1322 images

Unique patients:
  Total: 242
  MF: 149
  Non-MF: 93

Non-MF subtypes:
  B cell Lymphoma: 291 images (16 patients)
  PLEVA-PLC: 749 images (57 patients)
  T-cell dyscrasia: 122 images (9 patients)
  pseudolymphoma: 160 images (11 patients)


In [4]:
# Cell 3
PATCH_CACHE = Path('./patch_cache')
PATCH_CACHE.mkdir(exist_ok=True)

def extract_and_cache_patches(img_path, patch_size=512, stride=256, 
                              min_foreground_ratio=0.285, max_patches_per_image=200):
    key = hashlib.sha1(str(img_path).encode()).hexdigest()
    cache_dir = PATCH_CACHE / key
    if cache_dir.exists() and any(cache_dir.iterdir()):
        return sorted([str(p) for p in cache_dir.glob('*.jpg')])

    cache_dir.mkdir(parents=True, exist_ok=True)
    img = Image.open(img_path).convert('RGB')
    W,H = img.size
    patches = []

    for y in range(0, H-patch_size+1, stride):
        for x in range(0, W-patch_size+1, stride):
            crop = img.crop((x,y,x+patch_size,y+patch_size))
            arr = np.asarray(crop)
            
            # Convert to HSV and use saturation to find tissue
            hsv_img = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
            saturation = hsv_img[:, :, 1]
            fg_ratio = (saturation > 20).mean()

            if fg_ratio < min_foreground_ratio: continue
            
            fname = cache_dir / f'{x}_{y}.jpg'
            crop.save(fname, quality=90)
            patches.append(str(fname))
            if len(patches) >= max_patches_per_image: break
        if len(patches) >= max_patches_per_image: break
    return patches


In [5]:
# Cell 4
class MFHistologyDataset(Dataset):
    def __init__(self, rows, mag='x20', mode='train', patching=True, patch_size=512,
                 stride=256, transforms=None, max_patches_per_image=100):
        self.rows = [r for r in rows if (mag is None or r['mag']==mag)]
        self.mode = mode
        self.patching = patching
        self.patch_size = patch_size
        self.stride = stride
        self.max_patches_per_image = max_patches_per_image
        self.transforms = transforms
        labels = sorted(list({r['label'] for r in self.rows}))
        self.label2idx = {lab:i for i,lab in enumerate(labels)}


        self.items = []
        for r in self.rows:
            if self.patching:
                patches = extract_and_cache_patches(r['path'], patch_size=self.patch_size,
                                                    stride=self.stride, max_patches_per_image=self.max_patches_per_image)
                for p in patches:
                    self.items.append({'img': p, 'label': self.label2idx[r['label']], 'source': str(r['path'])})
            else:
                self.items.append({'img': str(r['path']), 'label': self.label2idx[r['label']], 'source': str(r['path'])})
        if len(self.items)==0:
            print("Warning: dataset empty for magnification", mag)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.open(it['img']).convert('RGB')
        if self.transforms: img = self.transforms(img)
        return img, it['label'], it['source']


train_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15), # transforms.RandomAffine(degrees=90, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10), intensive augmentation instead of rotation only
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((512,512), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [6]:
# Cell 5
def patient_split_stratified(rows, mag='x20', val_frac=0.15, seed=45):

    # group patients by class
    cls_map = {}
    for r in rows:
        # If mag is specified, filter by it. Otherwise, use all.
        if mag is not None and r['mag'] != mag: continue
        cls_map.setdefault(r['label'], {}).setdefault(r['patient'], []).append(r)

    # Now stratify
    train_rows, val_rows = [], []

    rng = random.Random(seed)

    for cls, patients_dict in cls_map.items():
        patients = list(patients_dict.keys())
        rng.shuffle(patients)

        n = len(patients)
        n_val = max(1, int(n * val_frac))

        val_p = set(patients[:n_val]) 

        for p, rlist in patients_dict.items():
            if p in val_p:  
                val_rows += rlist
            else: 
                train_rows += rlist
    return train_rows, val_rows



# Create a three-way split: Train (70%), Validation (15%), and Test (15%)
# First, split off a test set (15% of patients) from all data. `mag=None` considers all patients.
train_val_rows, test_rows = patient_split_stratified(all_images, mag=None, val_frac=0.15, seed=45)

# Now, split the remaining data into train and validation.
# The fraction for the validation set is 0.15 / (1 - 0.15) which is approx 0.1765
val_frac_adjusted = 0.15 / (1 - 0.15)
train_rows, val_rows = patient_split_stratified(train_val_rows, mag=None, val_frac=val_frac_adjusted, seed=45)

print(f"Total patients: {len(set(r['patient'] for r in all_images))}")
print(f"Training patients: {len(set(r['patient'] for r in train_rows))}")
print(f"Validation patients: {len(set(r['patient'] for r in val_rows))}")
print(f"Test patients: {len(set(r['patient'] for r in test_rows))}")


# --- Create Datasets for x20 ---
train_rows_20 = [r for r in train_rows if r['mag'] == 'x20']
val_rows_20 = [r for r in val_rows if r['mag'] == 'x20']
test_rows_20 = [r for r in test_rows if r['mag'] == 'x20'] 

train_ds_20 = MFHistologyDataset(train_rows_20, mag='x20', mode='train', patching=True, transforms=train_tf)
val_ds_20   = MFHistologyDataset(val_rows_20, mag='x20', mode='val', patching=True, transforms=val_tf)
test_ds_20  = MFHistologyDataset(test_rows_20, mag='x20', mode='test', patching=True, transforms=val_tf) 
print(f"\nx20 Datasets (patches): Train={len(train_ds_20)}, Val={len(val_ds_20)}, Test={len(test_ds_20)}")


# --- Create Datasets for x10 ---
train_rows_10 = [r for r in train_rows if r['mag'] == 'x10']
val_rows_10 = [r for r in val_rows if r['mag'] == 'x10']
test_rows_10 = [r for r in test_rows if r['mag'] == 'x10'] 

train_ds_10 = MFHistologyDataset(train_rows_10, mag='x10', mode='train', patching=True, transforms=train_tf)
val_ds_10   = MFHistologyDataset(val_rows_10, mag='x10', mode='val', patching=True, transforms=val_tf)
test_ds_10  = MFHistologyDataset(test_rows_10, mag='x10', mode='test', patching=True, transforms=val_tf)
print(f"x10 Datasets (patches): Train={len(train_ds_10)}, Val={len(val_ds_10)}, Test={len(test_ds_10)}")

def create_model(model_name='resnet50', pretrained=True, num_classes=2, dropout=0.2):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    return model


Total patients: 242
Training patients: 171
Validation patients: 36
Test patients: 35

x20 Datasets (patches): Train=88974, Val=18079, Test=15996
x10 Datasets (patches): Train=44865, Val=8669, Test=8077


In [9]:
# Cell 6 - Optuna Hyperparameter Tuning

# Configuration
OPTUNA_N_EPOCHS = 1
OPTUNA_N_TRIALS = 30
OPTUNA_DB = "optuna_mf.db"
SUBSAMPLE_RATIO = 0.15

def create_subsampled_dataset(dataset, ratio=0.15, seed=42):
    """Create subsampled dataset using Subset"""
    n_samples = len(dataset)
    n_subset = int(n_samples * ratio)
    indices = torch.randperm(n_samples, generator=torch.Generator().manual_seed(seed))[:n_subset].tolist()
    return Subset(dataset, indices)

def objective_fn(trial, train_subset, val_subset, mag, device):
    """Optuna objective function"""
    # Hyperparameters
    lr = trial.suggest_float('learning_rate', 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.1, 0.4)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    model_arch = trial.suggest_categorical('model_architecture', ['tf_efficientnet_b2', 'resnet50'])
    
    try:
        # Create model
        model = create_model(model_name=model_arch, pretrained=True, num_classes=2, dropout=dropout).to(device)
        
        # DataLoaders
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
        
        # Optimizer and criterion
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        # Compute class weights from subset
        counts = {}
        for idx in train_subset.indices:
            label = train_subset.dataset.items[idx]['label']
            counts[label] = counts.get(label, 0) + 1
        total = sum(counts.values())
        weights = torch.tensor([total/counts.get(i, 1) for i in range(2)], dtype=torch.float).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        
        scaler = torch.amp.GradScaler('cuda')
        
        # Training loop
        for epoch in range(OPTUNA_N_EPOCHS):
            # Train
            model.train()
            for imgs, labels, _ in train_loader:
                imgs = imgs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()
                with autocast('cuda'):
                    outputs = model(imgs)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            # Validate
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for imgs, labels, _ in val_loader:
                    imgs = imgs.to(device)
                    labels = labels.to(device)
                    outputs = model(imgs)
                    preds = outputs.argmax(dim=1)
                    correct += (preds == labels).sum().item()
                    total += imgs.size(0)
            
            val_acc = correct / total if total > 0 else 0.0
            
            # Report for pruning
            trial.report(val_acc, epoch)
            
            # Check pruning
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        return val_acc
        
    finally:
        # Clean up GPU memory
        del model
        if 'optimizer' in locals():
            del optimizer
        if 'criterion' in locals():
            del criterion
        if 'scaler' in locals():
            del scaler
        torch.cuda.empty_cache()
        gc.collect()

# Create subsampled datasets
print("\nCreating subsampled datasets for Optuna...")
train_subset_10 = create_subsampled_dataset(train_ds_10, ratio=SUBSAMPLE_RATIO, seed=42)
val_subset_10 = create_subsampled_dataset(val_ds_10, ratio=SUBSAMPLE_RATIO, seed=42)
train_subset_20 = create_subsampled_dataset(train_ds_20, ratio=SUBSAMPLE_RATIO, seed=42)
val_subset_20 = create_subsampled_dataset(val_ds_20, ratio=SUBSAMPLE_RATIO, seed=42)

print(f"x10 subsampled: Train={len(train_subset_10)}, Val={len(val_subset_10)}")
print(f"x20 subsampled: Train={len(train_subset_20)}, Val={len(val_subset_20)}")

# Pruner
pruner = MedianPruner(n_startup_trials=1, n_warmup_steps=0)

# Tune x10
print("\n" + "="*80)
print("TUNING x10 MODEL")
print("="*80)

storage_x10 = f"sqlite:///{OPTUNA_DB}"
study_x10 = optuna.create_study(
    study_name="mf_x10_tuning",
    storage=storage_x10,
    direction="maximize",
    pruner=pruner,
    load_if_exists=True
)

study_x10.optimize(
    lambda trial: objective_fn(trial, train_subset_10, val_subset_10, 'x10', 'cuda'),
    n_trials=OPTUNA_N_TRIALS,
    show_progress_bar=True
)

best_params_x10 = study_x10.best_params
best_value_x10 = study_x10.best_value

print(f"\nBest x10 validation accuracy: {best_value_x10:.4f}")
print(f"Best x10 parameters: {best_params_x10}")

# Save x10 results
with open('best_params_x10.json', 'w') as f:
    json.dump({
        'best_params': best_params_x10,
        'best_value': best_value_x10,
        'n_trials': len(study_x10.trials)
    }, f, indent=2)

# Clear GPU before x20
torch.cuda.empty_cache()
gc.collect()

# Tune x20
print("\n" + "="*80)
print("TUNING x20 MODEL")
print("="*80)

storage_x20 = f"sqlite:///{OPTUNA_DB}"
study_x20 = optuna.create_study(
    study_name="mf_x20_tuning",
    storage=storage_x20,
    direction="maximize",
    pruner=pruner,
    load_if_exists=True
)

study_x20.optimize(
    lambda trial: objective_fn(trial, train_subset_20, val_subset_20, 'x20', 'cuda'),
    n_trials=OPTUNA_N_TRIALS,
    show_progress_bar=True
)

best_params_x20 = study_x20.best_params
best_value_x20 = study_x20.best_value

print(f"\nBest x20 validation accuracy: {best_value_x20:.4f}")
print(f"Best x20 parameters: {best_params_x20}")

# Save x20 results
with open('best_params_x20.json', 'w') as f:
    json.dump({
        'best_params': best_params_x20,
        'best_value': best_value_x20,
        'n_trials': len(study_x20.trials)
    }, f, indent=2)

# Final cleanup
torch.cuda.empty_cache()
gc.collect()

print("\n" + "="*80)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*80)
print(f"\nx10 Best Config:")
print(f"  Validation Accuracy: {best_value_x10:.4f}")
for k, v in best_params_x10.items():
    print(f"  {k}: {v}")

print(f"\nx20 Best Config:")
print(f"  Validation Accuracy: {best_value_x20:.4f}")
for k, v in best_params_x20.items():
    print(f"  {k}: {v}")

print("\nSaved to:")
print("  - best_params_x10.json")
print("  - best_params_x20.json")
print(f"  - {OPTUNA_DB}")

[I 2026-01-04 19:27:54,189] Using an existing study with name 'mf_x10_tuning' instead of creating a new one.



Creating subsampled datasets for Optuna...
x10 subsampled: Train=6646, Val=1343
x20 subsampled: Train=13162, Val=2809

TUNING x10 MODEL


Best trial: 1. Best value: 0.640357:   3%|▎         | 1/30 [03:42<1:47:39, 222.74s/it]

[I 2026-01-04 19:31:36,931] Trial 1 finished with value: 0.6403574087862993 and parameters: {'learning_rate': 5.772006925209553e-05, 'weight_decay': 1.1700633777715577e-05, 'dropout': 0.24128006435313012, 'batch_size': 16, 'model_architecture': 'resnet50'}. Best is trial 1 with value: 0.6403574087862993.


Best trial: 1. Best value: 0.640357:   7%|▋         | 2/30 [06:55<1:35:43, 205.13s/it]

...

[I 2026-01-05 00:20:28,481] Trial 27 finished with value: 0.7276610893556426 and parameters: {'learning_rate': 3.194612308853048e-05, 'weight_decay': 0.0009685940749048608, 'dropout': 0.2078371935640277, 'batch_size': 8, 'model_architecture': 'resnet50'}. Best is trial 13 with value: 0.7390530437878249.


Best trial: 13. Best value: 0.739053:  97%|█████████▋| 29/30 [3:17:46<06:38, 398.38s/it]

[I 2026-01-05 00:27:22,169] Trial 28 finished with value: 0.7144891420434318 and parameters: {'learning_rate': 1.6500290992007425e-05, 'weight_decay': 0.0009016968598897439, 'dropout': 0.13815024154256783, 'batch_size': 8, 'model_architecture': 'resnet50'}. Best is trial 13 with value: 0.7390530437878249.


Best trial: 13. Best value: 0.739053: 100%|██████████| 30/30 [3:24:41<00:00, 409.38s/it]

[I 2026-01-05 00:34:17,250] Trial 29 pruned. 

Best x20 validation accuracy: 0.7391
Best x20 parameters: {'learning_rate': 3.2206153805902025e-05, 'weight_decay': 0.00021254307414870276, 'dropout': 0.2582534390245415, 'batch_size': 8, 'model_architecture': 'resnet50'}

HYPERPARAMETER TUNING COMPLETE

x10 Best Config:
  Validation Accuracy: 0.7036
  learning_rate: 3.950910421310083e-05
  weight_decay: 0.0009839492851857485
  dropout: 0.3979527105843245
  batch_size: 8
  model_architecture: resnet50

x20 Best Config:
  Validation Accuracy: 0.7391
  learning_rate: 3.2206153805902025e-05
  weight_decay: 0.00021254307414870276
  dropout: 0.2582534390245415
  batch_size: 8
  model_architecture: resnet50

Saved to:
  - best_params_x10.json
  - best_params_x20.json
  - optuna_mf.db


In [9]:
# Cell 7 - Final Training with Best Hyperparameters

# Save Path
save_path = Path("C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/Mycosis-Fungoides-Classifier/Trained Models")
save_path.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda")

# Load best hyperparameters from Optuna
print("\n" + "="*80)
print("LOADING BEST HYPERPARAMETERS")
print("="*80)


def compute_f2(preds, labels):
    """
    preds, labels: 1D torch tensors (class indices)
    """
    if len(preds) == 0:
        return 0.0
    return fbeta_score(
        labels.numpy(),
        preds.numpy(),
        beta=2,
        zero_division=0
    )

with open('best_params_x10.json', 'r') as f:
    best_config_x10 = json.load(f)
    best_params_x10 = best_config_x10['best_params']
    print(f"\nx10 Best Params (Val Acc: {best_config_x10['best_value']:.4f}):")
    for k, v in best_params_x10.items():
        print(f"  {k}: {v}")

with open('best_params_x20.json', 'r') as f:
    best_config_x20 = json.load(f)
    best_params_x20 = best_config_x20['best_params']
    print(f"\nx20 Best Params (Val Acc: {best_config_x20['best_value']:.4f}):")
    for k, v in best_params_x20.items():
        print(f"  {k}: {v}")

# Training functions
def train_one_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    total = 0
    correct = 0
    for imgs, labels, _src in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss/total, correct/total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += imgs.size(0)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    
    avg_loss = running_loss / total if total > 0 else 0
    acc = correct / total if total > 0 else 0

    all_preds = torch.cat(all_preds) if all_preds else torch.tensor([])
    all_labels = torch.cat(all_labels) if all_labels else torch.tensor([])

    f2 = compute_f2(all_preds, all_labels)
    return avg_loss, acc, f2

def get_class_weights(train_ds):
    counts = {}
    for it in train_ds.items:
        counts[it['label']] = counts.get(it['label'], 0) + 1
    total = sum(counts.values())
    weights = [total/counts.get(i, 1) for i in range(len(counts))]
    return torch.tensor(weights, dtype=torch.float).to(device)

# Create dataloaders with best batch sizes
train_loader_10 = DataLoader(train_ds_10, batch_size=best_params_x10['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
val_loader_10   = DataLoader(val_ds_10, batch_size=best_params_x10['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
test_loader_10  = DataLoader(test_ds_10, batch_size=best_params_x10['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

train_loader_20 = DataLoader(train_ds_20, batch_size=best_params_x20['batch_size'], shuffle=True, num_workers=0, pin_memory=True)
val_loader_20   = DataLoader(val_ds_20, batch_size=best_params_x20['batch_size'], shuffle=False, num_workers=0, pin_memory=True)
test_loader_20  = DataLoader(test_ds_20, batch_size=best_params_x20['batch_size'], shuffle=False, num_workers=0, pin_memory=True)

# Create models with best hyperparameters
model_10 = create_model(
    model_name=best_params_x10['model_architecture'], 
    pretrained=True, 
    num_classes=2, 
    dropout=best_params_x10['dropout']
).to(device)

model_20 = create_model(
    model_name=best_params_x20['model_architecture'], 
    pretrained=True, 
    num_classes=2, 
    dropout=best_params_x20['dropout']
).to(device)

# Compute class weights
weights_10 = get_class_weights(train_ds_10)
weights_20 = get_class_weights(train_ds_20)

# Loss functions
criterion_10 = nn.CrossEntropyLoss(weight=weights_10)
criterion_20 = nn.CrossEntropyLoss(weight=weights_20)

# Optimizers with best hyperparameters
optimizer_10 = optim.AdamW(
    model_10.parameters(), 
    lr=best_params_x10['learning_rate'], 
    weight_decay=best_params_x10['weight_decay']
)
optimizer_20 = optim.AdamW(
    model_20.parameters(), 
    lr=best_params_x20['learning_rate'], 
    weight_decay=best_params_x20['weight_decay']
)

# GradScalers
scaler_10 = torch.amp.GradScaler('cuda')
scaler_20 = torch.amp.GradScaler('cuda')

# Best best F₂ trackers
best_val_f2_10 = 0.0
best_val_f2_20 = 0.0

# Training configuration
EPOCHS = 9

print("\n" + "="*80)
print("STARTING FINAL TRAINING")
print("="*80)
print(f"Epochs: {EPOCHS}")
print(f"x10 Model: {best_params_x10['model_architecture']}")
print(f"x20 Model: {best_params_x20['model_architecture']}")

# Training loop
for epoch in range(EPOCHS):
    t0 = time.time()

    # Train x10 model
    train_loss_10, train_acc_10 = train_one_epoch(model_10, train_loader_10, optimizer_10, criterion_10, device, scaler_10)
    val_loss_10, val_acc_10, val_f2_10 = validate(model_10, val_loader_10, criterion_10, device)

    # Train x20 model
    train_loss_20, train_acc_20 = train_one_epoch(model_20, train_loader_20, optimizer_20, criterion_20, device, scaler_20)
    val_loss_20, val_acc_20, val_f2_20 = validate(model_20, val_loader_20, criterion_20, device)

    t1 = time.time()
    print(
    f"Epoch {epoch+1}/{EPOCHS} | "
    f"x10 acc {val_acc_10:.4f} F2 {val_f2_10:.4f} | "
    f"x20 acc {val_acc_20:.4f} F2 {val_f2_20:.4f} | "
    f"time {(t1-t0):.1f}s")

    # Save best models
    if val_f2_10 > best_val_f2_10:
        best_val_f2_10 = val_f2_10
        torch.save(model_10.state_dict(), save_path / f'model_{best_params_x10["model_architecture"]}_x10.pth')
        print(f"  ✓ Saved best x10 model (val_acc: {val_acc_10:.4f})")
    
    if val_f2_20 > best_val_f2_20:
        best_val_f2_20 = val_f2_20
        torch.save(model_20.state_dict(), save_path / f'model_{best_params_x20["model_architecture"]}_x20.pth')
        print(f"  ✓ Saved best x20 model (val_acc: {val_acc_20:.4f})")

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Best x10 validation F2: {best_val_f2_10:.4f}")
print(f"Best x20 validation F2: {best_val_f2_20:.4f}")
print(f"\nModels saved to: {save_path}")
print(f"  - model_{best_params_x10['model_architecture']}_x10.pth")
print(f"  - model_{best_params_x20['model_architecture']}_x20.pth")


LOADING BEST HYPERPARAMETERS

x10 Best Params (Val Acc: 0.7036):
  learning_rate: 3.950910421310083e-05
  weight_decay: 0.0009839492851857485
  dropout: 0.3979527105843245
  batch_size: 8
  model_architecture: resnet50

x20 Best Params (Val Acc: 0.7391):
  learning_rate: 3.2206153805902025e-05
  weight_decay: 0.00021254307414870276
  dropout: 0.2582534390245415
  batch_size: 8
  model_architecture: resnet50

STARTING FINAL TRAINING
Epochs: 9
x10 Model: resnet50
x20 Model: resnet50
Epoch 1/9 | x10 acc 0.6717 F2 0.4279 | x20 acc 0.7259 F2 0.6987 | time 4995.4s
  ✓ Saved best x10 model (val_acc: 0.6717)
  ✓ Saved best x20 model (val_acc: 0.7259)
Epoch 2/9 | x10 acc 0.6864 F2 0.5737 | x20 acc 0.6779 F2 0.6888 | time 4278.0s
  ✓ Saved best x10 model (val_acc: 0.6864)
Epoch 3/9 | x10 acc 0.6498 F2 0.6245 | x20 acc 0.6796 F2 0.6608 | time 4244.7s
  ✓ Saved best x10 model (val_acc: 0.6498)
Epoch 4/9 | x10 acc 0.6890 F2 0.5828 | x20 acc 0.7337 F2 0.6955 | time 4229.8s
Epoch 5/9 | x10 acc 0.657